In [1]:
import base64
import requests
import pandas as pd
import os
from tqdm import tqdm
import pickle

# OpenAI API Key
api_key = ""

In [2]:
df = pd.read_csv('../main.csv', index_col=0)

In [3]:
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

def decode_image(base64_string, output_image_path):
    image_data = base64.b64decode(base64_string)
    with open(output_image_path, "wb") as image_file:
        image_file.write(image_data)


In [ ]:
## JUST A QUIK TEST

# path to images
image_path = '/home/data/v.moskvoretskii/taxo_demo_images/images'
row_number = 0
item = df.loc[row_number]
idx = item['wordnet_id']

# Path to your image
path2image1 = f'{image_path}/{item["model_a"]}/{idx}.png'
if not f'{idx}.png' in os.listdir(f'{image_path}/{item["model_a"]}/'):
    path2image1 = f'{image_path}/{item["model_a"]}/{idx}.jpg'

    if not f'{idx}.jpg' in os.listdir(f'{image_path}/{item["model_a"]}/'):
        print('no such image')

# Path to your image
path2image2 = f'{image_path}/{item["model_b"]}/{idx}.png'
if not f'{idx}.png' in os.listdir(f'{image_path}/{item["model_b"]}/'):
    path2image2 = f'{image_path}/{item["model_b"]}/{idx}.jpg'
    if not f'{idx}.jpg' in os.listdir(f'{image_path}/{item["model_b"]}/'):
        print('no such image')


# Getting the base64 string
base64_image_a = encode_image(path2image1)
#decoding back
decode_image(base64_image_a, "output_image.jpg")
# make sure it is okay

# Getting the base64 string
base64_image_b = encode_image(path2image2)
#decoding back
decode_image(base64_image_b, "output_image2.jpg")
# make sure it is okay

In [7]:
item['core_lemma']

'coin'

In [8]:
preference_prompt = '''Please act as an impartial judge and evaluate the quality of the images provided by two AI image assistant to the user prompt displayed below.
You should choose the assistant that provide image which follows the users instructions better and reflects the users prompt main concept better.
Your evaluation should consider factors such as the image-text alignment, relevance, accuracy, depth, fidelty (overall image quality).
Begin your evaluation by comparing the two images and provide a short explanation.
Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision.
Do not allow the size of the images to influence your evaluation.
Do not favor certain names of the assistants.
Be as objective as possible.
After providing your explanation, output your final verdict by strictly following this format:
"[[A]]" if assistant A is better, "[[B]]" if assistant B is better, "[[C]]" for a tie and "[[D]]" if both images are bad.'''

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

payload = {
  "model": "gpt-4o-mini",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": f'{preference_prompt} \n [User Prompt] \n {item["core_lemma"]} [Start of first image]'
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image_a}"
          }
        },
        {
          "type": "text",
          "text": '[End of first image] \n [Start of second image]'
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image_b}"
          }
        },
        {
          "type": "text",
          "text": '[End of second image]'
        },
      ]
    }
  ],
  "max_tokens": 1024
}


In [9]:

response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

print(response.json())

{'error': {'code': 'unsupported_country_region_territory', 'message': 'Country, region, or territory not supported', 'param': None, 'type': 'request_forbidden'}}


# Если все ок то идем дальше

In [6]:
image_path = '/home/data/v.moskvoretskii/taxo_demo_images/images'

preference_prompt = '''Please act as an impartial judge and evaluate the quality of the images provided by two AI image assistant to the user prompt displayed below.
You should choose the assistant that provide image which follows the users instructions better and reflects the users prompt main concept better.
Your evaluation should consider factors such as the image-text alignment, relevance, accuracy, depth, fidelty (overall image quality).
Begin your evaluation by comparing the two images and provide a short explanation.
Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision.
Do not allow the size of the images to influence your evaluation.
Do not favor certain names of the assistants.
Be as objective as possible.
After providing your explanation, output your final verdict by strictly following this format:
"[[A]]" if assistant A is better, "[[B]]" if assistant B is better, "[[C]]" for a tie and "[[D]]" if both images are bad.'''

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

In [8]:
all_responses = []
for i, item in tqdm(df.iterrows()):
    idx = item['wordnet_id']

    # Path to your image
    path2image1 = f'{image_path}/{item["model_a"]}/{idx}.png'
    if not f'{idx}.png' in os.listdir(f'{image_path}/{item["model_a"]}/'):
        path2image1 = f'{image_path}/{item["model_a"]}/{idx}.jpg'

        if not f'{idx}.jpg' in os.listdir(f'{image_path}/{item["model_a"]}/'):
            print('no such image')
            all_responses.append('no such image')
            continue

    # Path to your image
    path2image2 = f'{image_path}/{item["model_b"]}/{idx}.png'
    if not f'{idx}.png' in os.listdir(f'{image_path}/{item["model_b"]}/'):
        path2image2 = f'{image_path}/{item["model_b"]}/{idx}.jpg'
        if not f'{idx}.jpg' in os.listdir(f'{image_path}/{item["model_b"]}/'):
            print('no such image')
            all_responses.append('no such image')
            continue


    # Getting the base64 string
    base64_image_a = encode_image(path2image1)
    base64_image_b = encode_image(path2image2)


    payload = {
        "model": "gpt-4o-mini",
        "messages": [
            {
            "role": "user",
            "content": [
                {
                "type": "text",
                "text": f'{preference_prompt} \n [User Prompt] {item["core_lemma"]} [Start of first image]'
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image_a}"
                }
                },
                {
                "type": "text",
                "text": '[End of first image] \n [Start of second image]'
                },
                {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image_b}"
                }
                },
                {
                "type": "text",
                "text": '[End of second image]'
                },
            ]
            }
        ],
        "max_tokens": 1024
    }

    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
    #response = payload
    all_responses.append(response)

10it [00:00, 98.42it/s]

553it [00:09, 17.18it/s] 

no such image


736it [00:18, 20.54it/s]

no such image


...

3059it [02:18, 18.36it/s]

no such image


3230it [02:26, 23.22it/s]

no such image


3370it [02:35, 21.66it/s]


In [9]:
len(all_responses), len(df)

(3370, 3370)

In [23]:
with open('gpt4_no_def_output.pickle', 'wb') as f:
    pickle.dump(all_responses, f)

In [4]:
with open('gpt4_no_def_output.pickle', 'rb') as f:
    ls = pickle.load(f)